# Anatomy of a QDE Decision

A step-by-step walkthrough of the Quantum Decision Engine's low-level API —
building belief states, applying operators, evolving through dynamics, and
collapsing to a decision.

**QDE flow:** context → belief state → evolving reasoning → decision

The high-level `AttoModel` wraps this pipeline behind `fit` / `predict`.
This notebook unpacks that pipeline using the low-level components directly:
`AttoState`, `AttoOperator`, `AttoDynamics`, `AttoMeasurement`, and `AttoEngine`.

This notebook demonstrates:
- **Belief state construction** — `uniform`, `from_probabilities`, `basis`, and `blend`
- **Operator mechanics** — `phase_shift`, `interference`, `rotation`, `from_signal`, and `compose`
- **State evolution** — `evolve`, `evolve_stepwise`, and `sensitivity`
- **Measurement** — `argmax`, `sample`, `softmax`, and `entropy`
- **AttoEngine** — the complete low-level pipeline in one object

In [1]:
# Environment check
import atto
print(f"atto version: {atto.__version__}")


atto version: 0.1.0


In [2]:
import numpy as np
from atto.core.state import AttoState
from atto.core.operator import AttoOperator
from atto.core.dynamics import AttoDynamics
from atto.core.measurement import AttoMeasurement

## 1 — Belief State Construction

A belief state is a complex amplitude vector in a Hilbert space. Each dimension
corresponds to a candidate strategy. The Born rule converts amplitudes to
probabilities: $P(i) = |\alpha_i|^2$.

Atto provides four ways to initialise a belief state.

In [3]:
labels = ["expand", "consolidate", "pivot"]

# 1. Uniform superposition — maximum uncertainty, all strategies equally likely
uniform = AttoState.uniform(3, labels=labels)
print("Uniform superposition (maximum uncertainty):")
print(f"  amplitudes:    {uniform.amplitudes}")
print(f"  probabilities: {uniform.probabilities}")
print(f"  dimension:     {uniform.dimension}")
print()

Uniform superposition (maximum uncertainty):
  amplitudes:    [0.57735027+0.j 0.57735027+0.j 0.57735027+0.j]
  probabilities: [0.33333333 0.33333333 0.33333333]
  dimension:     3



In [4]:
# 2. From a prior probability distribution — encode existing beliefs
prior = AttoState.from_probabilities([0.6, 0.3, 0.1])
print("From prior distribution [0.6, 0.3, 0.1]:")
print(f"  amplitudes:    {prior.amplitudes}")
print(f"  probabilities: {prior.probabilities}")
print()

From prior distribution [0.6, 0.3, 0.1]:
  amplitudes:    [0.77459667+0.j 0.54772256+0.j 0.31622777+0.j]
  probabilities: [0.6 0.3 0.1]



In [5]:
# 3. Basis state — full commitment to a single strategy
committed = AttoState.basis(0, dimension=3)
print("Basis state (full commitment to strategy 0):")
print(f"  amplitudes:    {committed.amplitudes}")
print(f"  probabilities: {committed.probabilities}")
print()

Basis state (full commitment to strategy 0):
  amplitudes:    [1.+0.j 0.+0.j 0.+0.j]
  probabilities: [1. 0. 0.]



In [6]:
# 4. Blend — coherent superposition of multiple belief states
# Two analysts have different beliefs. Blending preserves interference structure.
analyst_a = AttoState.from_probabilities([0.7, 0.2, 0.1])
analyst_b = AttoState.from_probabilities([0.1, 0.3, 0.6])

blended = AttoState.blend([analyst_a, analyst_b], weights=[0.6, 0.4])
print("Blended belief state (60% analyst A, 40% analyst B):")
print(f"  amplitudes:    {blended.amplitudes}")
print(f"  probabilities: {blended.probabilities}")
print()

# Note: blending amplitudes is NOT the same as averaging probabilities.
# Coherent superposition preserves interference between the two views.
naive_avg = 0.6 * np.array([0.7, 0.2, 0.1]) + 0.4 * np.array([0.1, 0.3, 0.6])
print(f"  naive prob avg:  {naive_avg}")
print(f"  blended probs:   {blended.probabilities}")
print("  → These differ because amplitude blending captures interference.")

Blended belief state (60% analyst A, 40% analyst B):
  amplitudes:    [0.6691537 +0.j 0.51895576+0.j 0.5319006 +0.j]
  probabilities: [0.44776667 0.26931508 0.28291825]

  naive prob avg:  [0.46 0.24 0.3 ]
  blended probs:   [0.44776667 0.26931508 0.28291825]
  → These differ because amplitude blending captures interference.


## 2 — Operators: How Signals Reshape Beliefs

An `AttoOperator` is a unitary matrix that transforms a belief state. Each
operator encodes how a signal reshapes the superposition of strategies.

Unitarity preserves the total probability (amplitudes remain normalised),
but redistributes it across strategies.

In [7]:
# Phase shift — rotates each strategy dimension independently.
# Positive phase boosts a strategy; negative suppresses it.
phase_op = AttoOperator.phase_shift(3, [0.5, 0.0, -0.3])
print("Phase shift operator [0.5, 0.0, -0.3]:")

state = AttoState.uniform(3, labels=labels)
evolved = phase_op.apply(state)
print(f"  before: {state.probabilities}")
print(f"  after:  {evolved.probabilities}")
print()

Phase shift operator [0.5, 0.0, -0.3]:
  before: [0.33333333 0.33333333 0.33333333]
  after:  [0.33333333 0.33333333 0.33333333]



In [8]:
# Interference — mixes two strategies via a Givens rotation.
# This is where the real reasoning happens: strategies interact.
intf_op = AttoOperator.interference(3, i=0, j=2, angle=0.4)
print("Interference operator (strategies 0 ↔ 2, angle=0.4):")

# Start from a state that favours strategy 0
state = AttoState.from_probabilities([0.7, 0.2, 0.1])
evolved = intf_op.apply(state)
print(f"  before: {state.probabilities}")
print(f"  after:  {evolved.probabilities}")
print("  → Probability transferred between strategies 0 and 2.")
print()

Interference operator (strategies 0 ↔ 2, angle=0.4):
  before: [0.7 0.2 0.1]
  after:  [0.41921743 0.2        0.38078257]
  → Probability transferred between strategies 0 and 2.



In [9]:
# Rotation — for two-strategy systems, a single-angle rotation
rot_op = AttoOperator.rotation(angle=0.8)
two_state = AttoState.from_probabilities([0.9, 0.1])
evolved = rot_op.apply(two_state)
print("Rotation (2D, angle=0.8):")
print(f"  before: {two_state.probabilities}")
print(f"  after:  {evolved.probabilities}")
print()

Rotation (2D, angle=0.8):
  before: [0.9 0.1]
  after:  [0.18844811 0.81155189]



In [10]:
# From signal — auto-generates a unitary operator from a raw context vector.
# This is how real-world signals enter the QDE pipeline.
signal = [0.1, -0.3, 0.5]
signal_op = AttoOperator.from_signal(signal)
print(f"Operator from signal {signal}:")

state = AttoState.uniform(3, labels=labels)
evolved = signal_op.apply(state)
print(f"  before: {state.probabilities}")
print(f"  after:  {evolved.probabilities}")
print()

Operator from signal [0.1, -0.3, 0.5]:
  before: [0.33333333 0.33333333 0.33333333]
  after:  [0.26711022 0.59983697 0.13305281]



In [11]:
# Compose — combine operators. Composition is non-commutative:
# op_a.compose(op_b) applies op_b first, then op_a.
op_a = AttoOperator.phase_shift(3, [0.5, 0.0, -0.3])
op_b = AttoOperator.interference(3, i=0, j=2, angle=0.4)

ab = op_a.compose(op_b)  # op_b first, then op_a
ba = op_b.compose(op_a)  # op_a first, then op_b

state = AttoState.uniform(3, labels=labels)
result_ab = ab.apply(state)
result_ba = ba.apply(state)

print("Non-commutativity of operator composition:")
print(f"  A then B: {result_ab.probabilities}")
print(f"  B then A: {result_ba.probabilities}")
print(f"  difference: {np.abs(result_ab.probabilities - result_ba.probabilities)}")
print("  → Order matters. This is non-commutativity in action.")

Non-commutativity of operator composition:
  A then B: [0.09421464 0.33333333 0.57245203]
  B then A: [0.16673773 0.33333333 0.49992893]
  difference: [7.25230965e-02 1.11022302e-16 7.25230965e-02]
  → Order matters. This is non-commutativity in action.


## 3 — Dynamics: Evolving the Belief State

`AttoDynamics` manages an ordered sequence of operators and evolves a state
through them. It also provides tools to inspect the trajectory and measure
how sensitive the outcome is to operator ordering.

In [12]:
dynamics = AttoDynamics()

# Build a reasoning sequence: market signal → internal assessment → competitive pressure
dynamics.add_operator(AttoOperator.from_signal([0.3, -0.1, 0.2]))   # market signal
dynamics.add_operator(AttoOperator.phase_shift(3, [0.1, 0.4, -0.2]))  # internal assessment
dynamics.add_operator(AttoOperator.interference(3, i=0, j=1, angle=0.5))  # competitive pressure

# Evolve the state through the full sequence
initial = AttoState.uniform(3, labels=labels)
evolved = dynamics.evolve(initial)

print("State evolution through 3 operators:")
print(f"  initial:  {initial.probabilities}")
print(f"  evolved:  {evolved.probabilities}")
print()

State evolution through 3 operators:
  initial:  [0.33333333 0.33333333 0.33333333]
  evolved:  [0.01254389 0.73508252 0.25237359]



In [13]:
# Stepwise evolution — inspect every intermediate state
trajectory = dynamics.evolve_stepwise(initial)

step_names = ["Initial", "After market signal", "After internal assessment", "After competitive pressure"]
print("Stepwise trajectory:")
for name, state in zip(step_names, trajectory):
    print(f"  {name:30s} → P = {state.probabilities}")
print()
print("Each step shows how the belief distribution shifts as new signals arrive.")

Stepwise trajectory:
  Initial                        → P = [0.33333333 0.33333333 0.33333333]
  After market signal            → P = [0.14511918 0.60250724 0.25237359]
  After internal assessment      → P = [0.14511918 0.60250724 0.25237359]
  After competitive pressure     → P = [0.01254389 0.73508252 0.25237359]

Each step shows how the belief distribution shifts as new signals arrive.


In [14]:
# Sensitivity — how much does operator ordering affect the final state?
# 0.0 = completely order-independent, higher = more order-sensitive
sensitivity = dynamics.sensitivity(initial)
print(f"Order sensitivity: {sensitivity:.4f}")
print()
if sensitivity > 0.01:
    print("→ The outcome is sensitive to signal ordering.")
    print("  Changing the sequence of evidence intake would produce a different decision.")
else:
    print("→ The outcome is approximately order-independent for this state.")

Order sensitivity: 0.0100

→ The outcome is sensitive to signal ordering.
  Changing the sequence of evidence intake would produce a different decision.


## 4 — Measurement: Collapsing to a Decision

`AttoMeasurement` collapses a belief state into a concrete decision. Three
methods are available, each suited to different use cases.

In [15]:
# Argmax — deterministic, picks the highest-probability strategy
m_argmax = AttoMeasurement(method="argmax")
decision = m_argmax.collapse(evolved)

print("Argmax measurement (deterministic):")
print(f"  action:        {decision.action}")
print(f"  label:         {decision.label}")
print(f"  confidence:    {decision.confidence:.4f}")
print(f"  probabilities: {decision.probabilities}")
print()

Argmax measurement (deterministic):
  action:        1
  label:         consolidate
  confidence:    0.7351
  probabilities: [0.01254389 0.73508252 0.25237359]



In [16]:
# Sample — stochastic, draws from the Born-rule distribution.
# Run multiple times to see the distribution in action.
m_sample = AttoMeasurement(method="sample")

samples = [m_sample.collapse(evolved).label for _ in range(100)]
print("Sampled measurement (100 draws from Born-rule distribution):")
for label in labels:
    count = samples.count(label)
    print(f"  {label:15s}: {count}/100 ({count/100:.2f})")
print(f"  expected probs:  {evolved.probabilities}")
print()

Sampled measurement (100 draws from Born-rule distribution):
  expand         : 1/100 (0.01)
  consolidate    : 73/100 (0.73)
  pivot          : 26/100 (0.26)
  expected probs:  [0.01254389 0.73508252 0.25237359]



In [17]:
# Softmax — temperature-controlled. Low temperature → more deterministic,
# high temperature → more exploratory.
for temp in [0.1, 0.5, 1.0, 2.0]:
    m_soft = AttoMeasurement(method="softmax", temperature=temp)
    d = m_soft.collapse(evolved)
    print(f"  T={temp:<4} → action={d.action} ({d.label}), confidence={d.confidence:.4f}")
print()

  T=0.1  → action=1 (consolidate), confidence=0.7351
  T=0.5  → action=1 (consolidate), confidence=0.7351
  T=1.0  → action=1 (consolidate), confidence=0.7351
  T=2.0  → action=1 (consolidate), confidence=0.7351



In [18]:
# Entropy — quantifies remaining uncertainty in a belief state.
# Tracks how uncertainty resolves through the pipeline.
print("Entropy through the evolution trajectory:")
for name, state in zip(step_names, trajectory):
    e = AttoMeasurement.entropy(state)
    print(f"  {name:30s} → entropy = {e:.4f}")
print()
print("→ Entropy decreasing means the belief state is resolving toward a decision.")
print("→ Entropy increasing means new signals introduced genuine ambiguity.")

Entropy through the evolution trajectory:
  Initial                        → entropy = 1.0986
  After market signal            → entropy = 0.9329
  After internal assessment      → entropy = 0.9329
  After competitive pressure     → entropy = 0.6286

→ Entropy decreasing means the belief state is resolving toward a decision.
→ Entropy increasing means new signals introduced genuine ambiguity.


## 5 — AttoEngine: The Complete Low-Level Pipeline

`AttoEngine` wraps state initialisation, operator sequencing, evolution, and
measurement into a single object. It is the low-level counterpart to `AttoModel`.

In [19]:
from atto.api.engine import AttoEngine

engine = AttoEngine(
    dimension=3,
    labels=["expand", "consolidate", "pivot"],
    measurement_method="argmax",
)

# Add signal operators to the evolution sequence
engine.add_operator(AttoOperator.phase_shift(3, [0.5, 0.0, -0.3]))
engine.add_operator(AttoOperator.interference(3, i=0, j=2, angle=0.4))
engine.add_operator(AttoOperator.from_signal([0.2, -0.1, 0.3]))

# Run the full pipeline: initialise → evolve → collapse
decision = engine.decide()

print("AttoEngine decision:")
print(f"  action:        {decision.action}")
print(f"  label:         {decision.label}")
print(f"  confidence:    {decision.confidence:.4f}")
print(f"  probabilities: {decision.probabilities}")

AttoEngine decision:
  action:        1
  label:         consolidate
  confidence:    0.4702
  probabilities: [0.11668117 0.47022769 0.41309114]


## Summary

This notebook walked through every layer of the QDE pipeline:

| Layer | Component | Role |
|-------|-----------|------|
| **State** | `AttoState` | Represents what is currently believed |
| **Operators** | `AttoOperator` | Encodes how signals reshape beliefs |
| **Dynamics** | `AttoDynamics` | Manages ordered evolution and sensitivity |
| **Measurement** | `AttoMeasurement` | Collapses belief into a decision |
| **Engine** | `AttoEngine` | Wraps the full pipeline |

Key takeaways:
- **Belief states are not feature vectors** — they are amplitude vectors where interference matters
- **Operators are not weights** — they are unitary transformations that preserve total probability
- **Order matters** — non-commutativity means the sequence of signals changes the outcome
- **Entropy tracks uncertainty** — it should generally decrease as evidence accumulates
- **Measurement is the only irreversible step** — everything before collapse is reversible evolution